# This code for we check  which stratagy is best for  our dataset:
## mean ,median for numaric
## Most frequent,constant for categorycal

In [93]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import FunctionTransformer

In [94]:
df=pd.read_csv('train.csv')

In [95]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [96]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)


In [118]:
x=df.drop(columns=['Survived'])
y=df['Survived']

In [104]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42) 

In [125]:
# Feature groups
numerical_features = ['Age', 'Fare']
categorical_features = ['Embarked', 'Sex']

# Pipelines
numerical_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transform, numerical_features),
    ('cat', categorical_transform, categorical_features)
])

# Main pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Grid Search
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],
    'classifier__C': [0.1, 1.0, 10, 100]
}

grid_search = GridSearchCV(clf, param_grid, cv=10)
grid_search.fit(x_train, y_train)
print(grid_search.best_params_)

{'classifier__C': 0.1, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}


###  here shows that  most frequent in catagorical data and mean at mumarical data is best here

In [126]:
print(f"internal cv score:{grid_search.best_score_:.3f}")

internal cv score:0.784


In [128]:
import pandas as pd
cv_result=pd.DataFrame(grid_search.cv_results_);
cv_result

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.023693,0.010347,0.009805,0.004934,0.1,most_frequent,mean,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
1,0.017313,0.001236,0.007892,0.000607,0.1,most_frequent,median,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
2,0.015564,0.002332,0.007218,0.000301,0.1,constant,mean,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
3,0.015302,0.000377,0.007211,0.000306,0.1,constant,median,"{'classifier__C': 0.1, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.774648,0.746479,0.887324,0.783725,0.079888,1
4,0.015497,0.001259,0.007255,0.000221,1.0,most_frequent,mean,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
5,0.015050,0.000211,0.007069,0.000073,1.0,most_frequent,median,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
6,0.015322,0.000203,0.007024,0.000025,1.0,constant,mean,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
7,0.015816,0.000314,0.007140,0.000270,1.0,constant,median,"{'classifier__C': 1.0, 'preprocessor__cat__imp...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
8,0.016372,0.001174,0.007649,0.000533,10.0,most_frequent,mean,"{'classifier__C': 10, 'preprocessor__cat__impu...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
9,0.017274,0.002141,0.008156,0.001694,10.0,most_frequent,median,"{'classifier__C': 10, 'preprocessor__cat__impu...",0.805556,0.75,0.690141,0.957746,0.802817,0.704225,0.71831,0.760563,0.746479,0.887324,0.782316,0.080160,5
